# Papiano Transcribe — Colab (GPU gratis)

Audio piano -> MIDI + MusicXML. GPU gratis, tanpa kartu, tanpa terminal.

## Cara pakai (gampang)
1. Menu atas: **Runtime -> Change runtime type -> T4 GPU -> Save**.
2. Menu atas: **Runtime -> Run all**.
3. Tunggu ~5 menit. Kalau muncul tombol **Restart**, klik, lalu **Run all** lagi.
4. Scroll ke sel paling bawah: muncul **form upload**. Drag audio -> Submit -> download MIDI + MusicXML.

Ga perlu nyentuh kode. Pipeline-nya sama persis dengan service produksi:
Aria-AMT (transkripsi) -> refine durasi (CQT) -> Beat This! (beat/downbeat) ->
quantize -> MusicXML 2 staff (music21).


In [ ]:
#@title 1. Install dependencies (~3-5 menit, restart runtime kalau diminta)
# Pin torch 2.5.0 supaya cocok dengan batasan aria-amt (torch>=2.3, torchaudio<=2.5).
!pip install -q torch==2.5.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "git+https://github.com/EleutherAI/aria-amt.git@a1ab73fc901d1759ec3bc173c146b3c6a3040261"
!pip install -q beat-this music21 librosa soundfile mido gradio
print("OK. Kalau Colab minta 'Restart runtime', restart lalu Run all lagi (install udah ke-cache, cepat).")


In [ ]:
#@title 2. Download checkpoint Aria-AMT (~426MB, publik, tanpa login)
import os, urllib.request

CKPT_DIR = "/content/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "piano-medium-double-1.0.safetensors")
URL = ("https://huggingface.co/datasets/loubb/aria-midi/resolve/main/"
       "piano-medium-double-1.0.safetensors?download=true")
if not os.path.exists(CKPT_PATH):
    print("Downloading ...")
    urllib.request.urlretrieve(URL, CKPT_PATH)
print("checkpoint:", CKPT_PATH, os.path.getsize(CKPT_PATH), "bytes")


In [ ]:
#@title 3. Pipeline (helpers + load model)
import base64, io, copy, logging, tempfile
import numpy as np
import torch

SAMPLE_RATE = 16000
MIN_NOTE_DURATION_S = 0.05
DECAY_THRESHOLD_RATIO = 0.15
TICKS_PER_BEAT = 480
GRID_SUBDIVISIONS = 4
DEFAULT_BPM = 120.0
STAFF_SPLIT = 60
MODEL_NAME = "medium-double"

logger = logging.getLogger("papiano")


def _refine_note_offsets(y, notes):
    import librosa
    hop_length = 256
    cqt = np.abs(librosa.cqt(y, sr=SAMPLE_RATE, hop_length=hop_length,
                             fmin=librosa.midi_to_hz(21), n_bins=88, bins_per_octave=12))
    frame_times = librosa.frames_to_time(np.arange(cqt.shape[1]), sr=SAMPLE_RATE, hop_length=hop_length)
    refined = []
    for note in notes:
        bin_idx = note["pitch"] - 21
        onset_idx = int(np.searchsorted(frame_times, note["onset"]))
        offset_idx = min(int(np.searchsorted(frame_times, note["offset"])), cqt.shape[1] - 1)
        if not (0 <= bin_idx < 88) or offset_idx <= onset_idx:
            refined.append(note); continue
        segment = cqt[bin_idx, onset_idx:offset_idx + 1]
        peak = segment.max()
        if peak <= 0:
            refined.append(note); continue
        threshold = peak * DECAY_THRESHOLD_RATIO
        peak_idx = int(segment.argmax())
        decay_idx = next((i for i in range(peak_idx, len(segment)) if segment[i] < threshold), None)
        if decay_idx is None:
            refined.append(note); continue
        new_offset = float(frame_times[onset_idx + decay_idx])
        new_offset = max(new_offset, note["onset"] + MIN_NOTE_DURATION_S)
        new_offset = min(new_offset, note["offset"])
        refined.append({**note, "offset": round(new_offset, 3)})
    return refined


def _beat_position_fn(beat_times):
    median_interval = float(np.median(np.diff(beat_times)))
    def pos(t):
        if t <= beat_times[0]:
            return (t - beat_times[0]) / median_interval
        if t >= beat_times[-1]:
            return (beat_times.size - 1) + (t - beat_times[-1]) / median_interval
        i = max(0, min(int(np.searchsorted(beat_times, t) - 1), beat_times.size - 2))
        span = beat_times[i + 1] - beat_times[i]
        return i + (t - beat_times[i]) / span if span > 0 else float(i)
    return pos


def _parse_time_signature(ts):
    if not ts:
        return None
    try:
        num = int(str(ts).split("/")[0])
    except (ValueError, IndexError):
        return None
    return num if 2 <= num <= 7 else None


def _apply_tempo_hint(beats, downbeats, tempo_hint, bpb_hint, audio_dur):
    if not tempo_hint or tempo_hint <= 0:
        return beats, downbeats
    if beats.size < 2:
        interval = 60.0 / tempo_hint
        beats = np.arange(0.0, audio_dur + interval, interval)
        downbeats = beats[::(bpb_hint or 4)]
        return beats, downbeats
    detected = 60.0 / float(np.median(np.diff(beats)))
    factors = np.array([0.25, 0.5, 1.0, 2.0, 4.0])
    f = float(factors[np.argmin(np.abs(detected * factors - tempo_hint))])
    if f > 1.0:
        n = int(round(f)); dense = []
        for a, b in zip(beats[:-1], beats[1:]):
            dense.extend(a + (b - a) * k / n for k in range(n))
        dense.append(float(beats[-1])); beats = np.asarray(dense, dtype=float)
    elif f < 1.0:
        beats = beats[::int(round(1.0 / f))]
    return beats, downbeats


def _infer_meter(beats, downbeats):
    if downbeats.size >= 2 and beats.size >= 2:
        counts = [int(np.sum((beats >= a - 1e-6) & (beats < b - 1e-6)))
                  for a, b in zip(downbeats[:-1], downbeats[1:])]
        counts = [c for c in counts if c > 0]
        if counts:
            bpb = int(round(np.median(counts)))
            if 2 <= bpb <= 7:
                return bpb
    return 4


def _build_midi(notes, pedals, bpm, beats, downbeats, bpb, quantize):
    import mido
    mid = mido.MidiFile(ticks_per_beat=TICKS_PER_BEAT)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(round(bpm)), time=0))
    track.append(mido.MetaMessage("time_signature", numerator=bpb, denominator=4, time=0))
    if quantize and beats.size >= 2:
        pos = _beat_position_fn(beats)
        origin = pos(float(downbeats[0])) if downbeats.size else 0.0
        all_onsets = [n["onset"] for n in notes] + [p["onset"] for p in pedals]
        min_pos = min((pos(t) for t in all_onsets), default=0.0)
        deficit = origin - min_pos
        pad_bars = int(np.ceil(deficit / bpb)) if deficit > 0 else 0
        shift = origin - pad_bars * bpb
        def to_tick(t, snap):
            bp = pos(t) - shift
            if snap:
                bp = round(bp * GRID_SUBDIVISIONS) / GRID_SUBDIVISIONS
            return max(0, round(bp * TICKS_PER_BEAT))
    else:
        beats_per_sec = bpm / 60.0
        def to_tick(t, snap):
            return max(0, round(t * beats_per_sec * TICKS_PER_BEAT))
    events = []
    for n in notes:
        on = to_tick(n["onset"], True); off = max(to_tick(n["offset"], True), on + 1)
        vel = max(1, min(127, int(n["velocity"])))
        events.append((on, 1, mido.Message("note_on", note=n["pitch"], velocity=vel)))
        events.append((off, 0, mido.Message("note_off", note=n["pitch"], velocity=0)))
    for p in pedals:
        on = to_tick(p["onset"], False); off = max(to_tick(p["offset"], False), on + 1)
        events.append((on, 2, mido.Message("control_change", control=64, value=127)))
        events.append((off, 0, mido.Message("control_change", control=64, value=0)))
    events.sort(key=lambda e: (e[0], e[1]))
    prev_tick = 0
    for tick, _g, msg in events:
        msg.time = tick - prev_tick; prev_tick = tick; track.append(msg)
    buf = io.BytesIO(); mid.save(file=buf); return buf.getvalue()


def _readable_sharps(sharps):
    alt = sharps - 12 if sharps > 0 else sharps + 12
    return alt if (-7 <= alt <= 7 and abs(alt) < abs(sharps)) else sharps


def _respell(part, use_flats):
    for n in part.recurse().notes:
        for p in n.pitches:
            if p.accidental is None:
                continue
            if use_flats and p.accidental.alter > 0:
                p.getHigherEnharmonic(inPlace=True)
            elif not use_flats and p.accidental.alter < 0:
                p.getLowerEnharmonic(inPlace=True)


def _build_musicxml(midi_bytes, beats_per_bar):
    import music21
    with tempfile.NamedTemporaryFile(suffix=".mid") as tmp_mid:
        tmp_mid.write(midi_bytes); tmp_mid.flush()
        score_in = music21.converter.parse(tmp_mid.name)
    try:
        analyzed = score_in.analyze("key")
        sharps = _readable_sharps(analyzed.sharps); mode = analyzed.mode
    except Exception:
        sharps, mode = 0, "major"
    use_flats = sharps < 0
    key_name = music21.key.KeySignature(sharps).asKey(mode).name
    ts_str = f"{beats_per_bar}/4"
    parts = {}
    for hand, clef in (("RH", music21.clef.TrebleClef()), ("LH", music21.clef.BassClef())):
        part = music21.stream.Part(); part.partName = f"Piano ({hand})"
        part.insert(0, clef); part.insert(0, music21.meter.TimeSignature(ts_str))
        part.insert(0, music21.key.KeySignature(sharps)); parts[hand] = part
    for el in score_in.flatten().notes:
        top_pitch = max(p.midi for p in el.pitches)
        parts["RH" if top_pitch >= STAFF_SPLIT else "LH"].insert(el.offset, copy.deepcopy(el))
    for part in parts.values():
        _respell(part, use_flats); part.makeMeasures(inPlace=True); part.makeRests(fillGaps=True, inPlace=True)
    score_out = music21.stream.Score()
    score_out.insert(0, parts["RH"]); score_out.insert(0, parts["LH"])
    score_out.insert(0, music21.layout.StaffGroup([parts["RH"], parts["LH"]], symbol="brace", barTogether=True))
    xml = music21.musicxml.m21ToXml.GeneralObjectExporter(score_out).parse()
    return xml.decode("utf-8"), key_name


# ---- load models once ----
from amt.audio import AudioTransform
from amt.config import load_model_config
from amt.inference.model import AmtEncoderDecoder, ModelConfig
from amt.tokenizer import AmtTokenizer
from amt.utils import _load_weight

TOKENIZER = AmtTokenizer()
_cfg = ModelConfig(**load_model_config(MODEL_NAME)); _cfg.set_vocab_size(TOKENIZER.vocab_size)
MODEL = AmtEncoderDecoder(_cfg)
_state = _load_weight(ckpt_path=CKPT_PATH)
_state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in _state.items()}
MODEL.load_state_dict(_state)
MODEL.decoder.setup_cache(batch_size=1, max_seq_len=4096,
                          dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float)
MODEL.cuda(); MODEL.eval()
AUDIO_TRANSFORM = AudioTransform().cuda()

from beat_this.inference import File2Beats
BEAT_TRACKER = File2Beats(checkpoint_path="final0", device="cuda" if torch.cuda.is_available() else "cpu")
print("models loaded")


In [ ]:
#@title 4. Fungsi transkripsi
import soundfile as sf, librosa


def transcribe_audio(audio_path, quantize=True, tempo_hint=None, time_signature=None):
    from amt.data import get_wav_segments
    from amt.inference.transcribe import (
        CHUNK_LEN_MS, LEN_MS, STRIDE_FACTOR, _get_silent_intervals,
        _process_silent_intervals, _shift_onset, _truncate_seq, process_segments)
    tokenizer = TOKENIZER
    seq = [tokenizer.bos_tok]; concat_seq = [tokenizer.bos_tok]; idx = 0
    for curr_audio_segment in get_wav_segments(audio_path=audio_path, stride_factor=STRIDE_FACTOR, pad_last=True):
        init_idx = len(seq)
        silent_intervals = _get_silent_intervals(curr_audio_segment)
        input_seq = list(seq)
        results = process_segments(tasks=[((curr_audio_segment, seq), 0)], model=MODEL,
                                   audio_transform=AUDIO_TRANSFORM, tokenizer=tokenizer, logger=logger)
        seq = results[0]
        seq_adj = _process_silent_intervals(seq, intervals=silent_intervals, tokenizer=tokenizer)
        if len(seq_adj) < len(seq) - 15:
            seq = seq_adj
        try:
            next_seq = _truncate_seq(seq, CHUNK_LEN_MS, LEN_MS - CHUNK_LEN_MS)
        except Exception:
            try:
                seq = _truncate_seq(input_seq, CHUNK_LEN_MS - 2, CHUNK_LEN_MS)
            except Exception:
                seq = [tokenizer.bos_tok]
        else:
            if seq[-1] == tokenizer.eos_tok:
                seq = seq[:-1]
            concat_seq += _shift_onset(seq[init_idx:], idx * CHUNK_LEN_MS)
            seq = [tokenizer.bos_tok] if len(next_seq) == 1 else next_seq
        idx += 1

    last_onset = next(tok[1] for tok in reversed(concat_seq) if isinstance(tok, tuple) and tok[0] == "onset")
    midi_dict = tokenizer.detokenize(concat_seq, last_onset)
    midi_dict.remove_redundant_pedals()

    notes = [{"pitch": int(m["data"]["pitch"]), "onset": m["data"]["start"] / 1000.0,
              "offset": m["data"]["end"] / 1000.0, "velocity": int(m["data"]["velocity"])}
             for m in midi_dict.note_msgs]

    pedals = []; pedal_on = None
    for m in sorted(midi_dict.pedal_msgs, key=lambda m: m["tick"]):
        if m["data"] == 1 and pedal_on is None:
            pedal_on = m["tick"]
        elif m["data"] == 0 and pedal_on is not None:
            pedals.append({"onset": pedal_on / 1000.0, "offset": m["tick"] / 1000.0}); pedal_on = None
    if pedal_on is not None:
        pedals.append({"onset": pedal_on / 1000.0, "offset": last_onset / 1000.0})

    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    notes = _refine_note_offsets(y, notes)

    beats = downbeats = np.asarray([], dtype=float)
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav") as tw:
            sf.write(tw.name, y, SAMPLE_RATE)
            bt_beats, bt_downbeats = BEAT_TRACKER(tw.name)
        beats = np.asarray(bt_beats, dtype=float); downbeats = np.asarray(bt_downbeats, dtype=float)
    except Exception:
        logger.exception("Beat This! failed")

    bpb_hint = _parse_time_signature(time_signature)
    beats, downbeats = _apply_tempo_hint(beats, downbeats, tempo_hint, bpb_hint, len(y) / SAMPLE_RATE)
    if beats.size >= 2:
        bpm = 60.0 / float(np.median(np.diff(beats)))
        if not np.isfinite(bpm) or bpm <= 0:
            bpm = tempo_hint or DEFAULT_BPM
    else:
        bpm = tempo_hint or DEFAULT_BPM
    bpb = bpb_hint or _infer_meter(beats, downbeats)

    midi_bytes = _build_midi(notes, pedals, bpm, beats, downbeats, bpb, quantize)
    musicxml, key = None, None
    try:
        musicxml, key = _build_musicxml(midi_bytes, bpb)
    except Exception:
        logger.exception("MusicXML failed")
    return {"notes": notes, "pedals": pedals, "tempo": round(bpm, 1),
            "time_signature": f"{bpb}/4", "key": key, "midi_bytes": midi_bytes, "musicxml": musicxml}


In [ ]:
#@title 5. APP — form upload (jalan otomatis abis Run all)
import os, tempfile
import gradio as gr


def _run(audio_path, tempo_hint, time_signature):
    if not audio_path:
        return None, None, "Upload audio dulu."
    hint = tempo_hint if (tempo_hint and tempo_hint > 0) else None
    ts = time_signature or None
    res = transcribe_audio(audio_path, quantize=True, tempo_hint=hint, time_signature=ts)
    stem = os.path.splitext(os.path.basename(audio_path))[0]
    out_dir = tempfile.mkdtemp()
    mid_path = os.path.join(out_dir, stem + ".mid")
    with open(mid_path, "wb") as f:
        f.write(res["midi_bytes"])
    xml_path = None
    if res["musicxml"]:
        xml_path = os.path.join(out_dir, stem + ".musicxml")
        with open(xml_path, "w") as f:
            f.write(res["musicxml"])
    info = (f"notes={len(res['notes'])} | pedals={len(res['pedals'])} | "
            f"tempo={res['tempo']} BPM | {res['time_signature']} | key={res['key']}")
    return mid_path, xml_path, info


demo = gr.Interface(
    fn=_run,
    inputs=[
        gr.Audio(type="filepath", label="Audio piano (mp3/wav)"),
        gr.Number(value=0, label="Tempo hint BPM (0 = auto; rubato/nocturne isi mis. 50)"),
        gr.Dropdown(choices=["", "4/4", "3/4", "2/4", "6/4"], value="",
                    label="Time signature (kosong = auto)"),
    ],
    outputs=[
        gr.File(label="MIDI (.mid)"),
        gr.File(label="Sheet music (.musicxml) -> buka di MuseScore"),
        gr.Textbox(label="Info"),
    ],
    title="Papiano Transcribe",
    description="Drag audio piano, klik Submit. Hasil MIDI + sheet music bisa di-download.",
)
# share=True kasih link publik (xxx.gradio.live) yang bisa dibuka di browser/HP.
demo.launch(share=True)
